# Notebook 09 — Class Imbalance and Metrics That Actually Matter

Real-world datasets are almost never balanced. Medical imaging might be 99% healthy and 1% disease. Fraud detection is 99.9% legitimate transactions. Defect inspection is 99% good parts. If you train naïvely and measure with plain accuracy, you get a model that looks great in a report and is useless in production.

This notebook builds an imbalanced version of CIFAR-10 (which we will call **CIFAR-10-IMB**), reproduces the standard failure mode, and walks through the four techniques every practitioner should know: **better metrics**, **stratified splits**, **class-weighted loss**, **weighted sampling**, and **focal loss**. We finish with **threshold tuning** for binary problems.

## Learning goals
- Understand why accuracy is a dangerous metric on imbalanced data.
- Use precision / recall / F1-macro / balanced accuracy / ROC-AUC / PR-AUC.
- Make **stratified** train/val splits.
- Apply three re-balancing techniques (class weights, weighted sampler, focal loss) and compare.
- Tune the decision threshold for a binary problem to hit a recall target.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/09_class_imbalance_and_metrics.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Build CIFAR-10-IMB (synthetic imbalance)

CIFAR-10 has 5000 training images for each of 10 classes. Real datasets are rarely so kind. We simulate imbalance by keeping classes 0-4 (airplane, automobile, bird, cat, deer) at full size and downsampling classes 5-9 (dog, frog, horse, ship, truck) to **1/20th** of their original count — so ~250 images each.

Result: a long-tailed dataset where the majority:minority ratio is 20:1. This is mild compared to real imbalance (often 100:1 or 1000:1) but enough to break a naïve trainer.

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Subset, DataLoader
import numpy as np
from collections import Counter
from utils.env import data_dir

# CIFAR-10 is small (~170MB) and downloads quickly.
CIFAR_ROOT = data_dir()

# Minimal transforms for speed — we care about method comparisons, not SOTA here.
train_tf = transforms.Compose([
    transforms.Resize(64),                 # mild upscale so conv stacks have room
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_tf = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

full_train = torchvision.datasets.CIFAR10(CIFAR_ROOT, train=True, download=True, transform=train_tf)
full_test  = torchvision.datasets.CIFAR10(CIFAR_ROOT, train=False, download=True, transform=eval_tf)

targets = np.array(full_train.targets)
print('Original class counts:', Counter(full_train.targets))

In [ ]:
# Keep classes 0-4 at full size, shrink classes 5-9 to 1/20th.
MAJOR_CLASSES = [0, 1, 2, 3, 4]
MINOR_CLASSES = [5, 6, 7, 8, 9]
MINOR_FRAC = 1 / 20

rng = np.random.default_rng(42)
imb_indices = []
for c in range(10):
    idx = np.where(targets == c)[0]
    if c in MINOR_CLASSES:
        keep = max(1, int(len(idx) * MINOR_FRAC))
        idx = rng.choice(idx, size=keep, replace=False)
    imb_indices.extend(idx.tolist())
imb_indices = np.array(imb_indices)

imb_targets = targets[imb_indices]
print('CIFAR-10-IMB class counts:', dict(sorted(Counter(imb_targets.tolist()).items())))
print('Total samples:', len(imb_indices))
print(f'Ratio majority:minority = {Counter(imb_targets.tolist())[0] / Counter(imb_targets.tolist())[5]:.1f}:1')

## 2. Stratified train/val split

A random 80/20 split on imbalanced data might miss the minority classes entirely in the val set (or pack them all into one side). `StratifiedShuffleSplit` keeps the class ratio constant across both splits. **Always use stratified splits for classification.**

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

# Naive random split:
rand_tr_idx, rand_va_idx = train_test_split(
    np.arange(len(imb_indices)), test_size=0.2, random_state=0
)
print('--- naive random split ---')
print('  train:', dict(sorted(Counter(imb_targets[rand_tr_idx].tolist()).items())))
print('  val  :', dict(sorted(Counter(imb_targets[rand_va_idx].tolist()).items())))

# Stratified split: class fractions preserved.
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
strat_tr_idx, strat_va_idx = next(sss.split(np.zeros(len(imb_targets)), imb_targets))
print('--- stratified split ---')
print('  train:', dict(sorted(Counter(imb_targets[strat_tr_idx].tolist()).items())))
print('  val  :', dict(sorted(Counter(imb_targets[strat_va_idx].tolist()).items())))

# Materialize the actual datasets using original full_train indices.
train_ds = Subset(full_train, imb_indices[strat_tr_idx].tolist())

# Val set uses *eval* transforms, so we rebuild it from a fresh CIFAR instance.
val_src = torchvision.datasets.CIFAR10(CIFAR_ROOT, train=True, download=False, transform=eval_tf)
val_ds  = Subset(val_src, imb_indices[strat_va_idx].tolist())

train_labels = imb_targets[strat_tr_idx]
val_labels   = imb_targets[strat_va_idx]
print(f'\ntrain size={len(train_ds)}, val size={len(val_ds)}')

## 3. A tiny baseline CNN

We will train the *same small model* five times with different balancing strategies, so keep this simple and fast. A 3-block CNN with global pool is enough to show the effect.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        return self.fc(x.flatten(1))

def fresh_model():
    torch.manual_seed(0)
    return TinyCNN().to(device)

print(fresh_model())

## 4. A small training loop we can reuse

We deliberately train for only a handful of epochs — the goal here is to compare techniques under equal conditions, not to maximize absolute accuracy.

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

def make_loaders(train_ds, val_ds, train_sampler=None, batch_size=128):
    shuffle = train_sampler is None
    train_dl = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, shuffle=shuffle, num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0)
    return train_dl, val_dl

def train_and_eval(model, train_dl, val_dl, loss_fn, epochs=4, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
        print(f'  epoch {ep+1}/{epochs}  last batch loss={loss.item():.3f}')
    # Collect predictions
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            all_logits.append(model(xb.to(device)).cpu())
            all_y.append(yb)
    logits = torch.cat(all_logits)
    y_true = torch.cat(all_y).numpy()
    y_pred = logits.argmax(dim=1).numpy()
    probs  = torch.softmax(logits, dim=1).numpy()
    return y_true, y_pred, probs

print('helpers ready')

## 5. Why imbalance is a problem (the naïve baseline)

Train with plain `CrossEntropyLoss`. Overall accuracy looks respectable, but **per-class** accuracy reveals the model gave up on the minority classes.

In [ ]:
model = fresh_model()
train_dl, val_dl = make_loaders(train_ds, val_ds)
y_true_b, y_pred_b, probs_b = train_and_eval(model, train_dl, val_dl, nn.CrossEntropyLoss(), epochs=4)

print(f'\nOverall accuracy: {accuracy_score(y_true_b, y_pred_b):.3f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_true_b, y_pred_b):.3f}')
print(f'F1 (macro):        {f1_score(y_true_b, y_pred_b, average="macro"):.3f}')

# Per-class recall = diag(cm_norm_by_row)
cm = confusion_matrix(y_true_b, y_pred_b, labels=list(range(10)))
per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
for c, r in enumerate(per_class_recall):
    tag = 'majority' if c in MAJOR_CLASSES else 'minority'
    print(f'  class {c} ({tag}): recall={r:.2f}')

## 6. Proper metrics for imbalance

**Accuracy** = (correct) / (total). Dominated by whichever class has the most samples.

**Balanced accuracy** = mean of per-class recall. Every class contributes equally.

**F1-macro** = unweighted mean of F1 across classes. Penalizes models that ignore rare classes.

**ROC-AUC (OvR)** = for each class, area under the TPR/FPR curve. Robust to class ratios.

**PR-AUC (average precision)** = area under precision/recall curve. Preferred to ROC-AUC when the positive class is very rare (ROC-AUC can look misleadingly high).

In [ ]:
from utils.metrics import classification_report_dict
import pandas as pd

def summary_row(name, y_true, y_pred, probs):
    return {
        'method': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'roc_auc_ovr': roc_auc_score(y_true, probs, multi_class='ovr', labels=list(range(10))),
    }

rows = [summary_row('baseline CE', y_true_b, y_pred_b, probs_b)]

# Also print the per-class classification report
rep = classification_report_dict(y_true_b, y_pred_b, labels=[str(i) for i in range(10)])
print('Per-class (baseline):')
for k, v in rep.items():
    if k.isdigit():
        print(f"  class {k}: precision={v['precision']:.2f}  recall={v['recall']:.2f}  f1={v['f1-score']:.2f}")

## 7. Technique 1 — class weights in CrossEntropyLoss

The simplest fix: tell the loss to up-weight mistakes on rare classes. A common formula is

$$w_c = \frac{N}{K \cdot n_c}$$

where $N$ = total samples, $K$ = number of classes, $n_c$ = samples in class $c$. Pass as the `weight=` kwarg of `nn.CrossEntropyLoss`.

In [ ]:
class_counts = np.array([int((train_labels == c).sum()) for c in range(10)])
N, K = class_counts.sum(), len(class_counts)
weights = N / (K * class_counts)
weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
print('class weights:', np.round(weights, 2))

model = fresh_model()
train_dl, val_dl = make_loaders(train_ds, val_ds)
y_true_w, y_pred_w, probs_w = train_and_eval(
    model, train_dl, val_dl,
    nn.CrossEntropyLoss(weight=weights_t),
    epochs=4,
)
print(f'\nbalanced_acc={balanced_accuracy_score(y_true_w, y_pred_w):.3f}  '
      f'f1_macro={f1_score(y_true_w, y_pred_w, average="macro"):.3f}')
rows.append(summary_row('weighted CE', y_true_w, y_pred_w, probs_w))

## 8. Technique 2 — WeightedRandomSampler (oversampling)

Instead of reweighting the loss, re-balance the data itself: draw more samples from rare classes during training. Sample weight per example is $1 / n_{y_i}$. `WeightedRandomSampler` implements exactly this, plugs into `DataLoader`, and costs nothing at inference.

In [ ]:
from torch.utils.data import WeightedRandomSampler

sample_weights = 1.0 / class_counts[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),   # one “epoch” draws as many samples as there are images
    replacement=True,
)

model = fresh_model()
train_dl, val_dl = make_loaders(train_ds, val_ds, train_sampler=sampler)
y_true_s, y_pred_s, probs_s = train_and_eval(model, train_dl, val_dl, nn.CrossEntropyLoss(), epochs=4)
print(f'\nbalanced_acc={balanced_accuracy_score(y_true_s, y_pred_s):.3f}  '
      f'f1_macro={f1_score(y_true_s, y_pred_s, average="macro"):.3f}')
rows.append(summary_row('weighted sampler', y_true_s, y_pred_s, probs_s))

## 9. Technique 3 — Focal Loss

**Focal Loss** (Lin et al., 2017, *Focal Loss for Dense Object Detection*) down-weights examples the model already gets right. Definition:

$$\mathrm{FL}(p_t) = -(1 - p_t)^{\gamma} \log(p_t)$$

where $p_t$ is the predicted probability of the true class and $\gamma \ge 0$ is the *focusing parameter*. At $\gamma=0$ it reduces to cross-entropy. At $\gamma=2$ (paper default), an easy example with $p_t=0.9$ contributes $(1-0.9)^2 = 0.01$ of its usual loss. Hard examples (often minority-class) dominate the gradient.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: torch.Tensor | None = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha   # optional per-class weights; complements the focusing term
    def forward(self, logits, targets):
        # log_softmax is numerically safer than log(softmax)
        logp = F.log_softmax(logits, dim=1)              # (B, K)
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)  # log p_t
        p_t = logp_t.exp()
        focal = (1 - p_t) ** self.gamma * (-logp_t)
        if self.alpha is not None:
            focal = focal * self.alpha[targets]
        return focal.mean()

# gamma=0 sanity check -> close to plain CE
x = torch.randn(8, 10); y = torch.randint(0, 10, (8,))
print('CE        :', F.cross_entropy(x, y).item())
print('Focal g=0 :', FocalLoss(gamma=0)(x, y).item())
print('Focal g=2 :', FocalLoss(gamma=2)(x, y).item())

In [ ]:
model = fresh_model()
train_dl, val_dl = make_loaders(train_ds, val_ds)
y_true_f, y_pred_f, probs_f = train_and_eval(
    model, train_dl, val_dl,
    FocalLoss(gamma=2.0),
    epochs=4,
)
print(f'\nbalanced_acc={balanced_accuracy_score(y_true_f, y_pred_f):.3f}  '
      f'f1_macro={f1_score(y_true_f, y_pred_f, average="macro"):.3f}')
rows.append(summary_row('focal (g=2)', y_true_f, y_pred_f, probs_f))

## 10. Head-to-head comparison

Raw accuracy changes barely at all — but *balanced accuracy* and *F1-macro* tell the real story. Each rebalancing technique trades a small amount of majority-class accuracy for a large gain on minority classes.

In [ ]:
df = pd.DataFrame(rows).set_index('method').round(3)
df

## 11. Binary metrics deep dive

For intuition let's recast the problem as *binary*: class 0 (airplane) vs everything else. Now we can draw ROC and PR curves.

In [ ]:
from utils.plotting import plot_roc_pr

# Use the class-weighted model's probabilities.
y_true_bin = (y_true_w == 0).astype(int)
scores_bin = probs_w[:, 0]

roc = roc_auc_score(y_true_bin, scores_bin)
pr  = average_precision_score(y_true_bin, scores_bin)
print(f'ROC-AUC = {roc:.3f}')
print(f'PR-AUC  = {pr:.3f}  (aka average precision)')

plot_roc_pr(y_true_bin, scores_bin, title='Class 0 vs rest');

## 12. Threshold tuning

In production, the operating point matters more than the curve. The default `argmax` threshold of 0.5 is arbitrary. Depending on business cost, you might want the threshold that
- maximizes F1, or
- achieves at least 95% recall (e.g. medical screening), or
- achieves at least 99% precision (e.g. fraud block list).

Sweep thresholds on the validation scores, then pick.

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_true_bin, scores_bin)
# sklearn returns thresholds aligned with prec[:-1], rec[:-1]
f1 = 2 * prec * rec / (prec + rec + 1e-12)

# 1) Threshold that maximizes F1.
i_best = np.nanargmax(f1[:-1])
print(f'Best-F1 threshold: {thr[i_best]:.3f}  '
      f'(precision={prec[i_best]:.2f}, recall={rec[i_best]:.2f}, f1={f1[i_best]:.2f})')

# 2) Smallest threshold that still hits recall >= 0.95.
mask = rec[:-1] >= 0.95
if mask.any():
    i = np.argmax(mask[::-1])   # last index meeting the criterion
    i = len(mask) - 1 - i
    print(f'Recall>=0.95 threshold: {thr[i]:.3f}  '
          f'(precision={prec[i]:.2f}, recall={rec[i]:.2f})')
else:
    print('No threshold achieves recall >= 0.95 on this val set.')

## 13. Per-class confusion matrix

Normalize by row so each entry shows *“given a true class-X image, what fraction got labeled class-Y?”*. Off-diagonal hot spots are the classes your model confuses.

In [ ]:
from utils.plotting import plot_confusion_matrix

class_names = ['airplane','auto','bird','cat','deer','dog','frog','horse','ship','truck']
plot_confusion_matrix(y_true_w, y_pred_w, class_names=class_names, normalize=True, title='Weighted-CE model');

## Key Takeaways

- **Accuracy lies** on imbalanced data. Report balanced accuracy, F1-macro, and PR-AUC.
- **Always stratify** your train/val/test splits for classification.
- Three common fixes:
  1. Class-weighted `CrossEntropyLoss` — fastest to try.
  2. `WeightedRandomSampler` — effective, zero inference cost.
  3. Focal loss — $\gamma=2$ is the battle-tested default.
- These techniques are **not mutually exclusive**; you can combine class-weighted focal loss with a weighted sampler.
- For binary decisions, **tune the threshold** to your business metric; don't rely on 0.5.
- Always inspect the **confusion matrix** — it exposes which class-pairs the model confuses and is often more actionable than a single scalar.

## Exercises

1. Make CIFAR-10-IMB *harder*: keep classes 0-4 full and reduce classes 5-9 to 1/100th. Rerun the comparison table. Which method is most robust to extreme imbalance?
2. Combine class-weighted focal loss ($\alpha = w_c$ above, $\gamma=2$) with a weighted sampler. Does it help further?
3. Implement **label smoothing** (`CrossEntropyLoss(label_smoothing=0.1)`) and see whether it helps minority-class F1.
4. Swap the TinyCNN for a timm `efficientnetv2_rw_t` with pretrained ImageNet weights. How much does transfer learning shift the balanced-accuracy leaderboard?
5. Pick a different recall target (e.g. 99%) and find the implied precision cost. What threshold would you ship with?